# ML Pipeline 3: Reintegration Readiness
Predictive + explanatory models for resident readiness progression.

In [ ]:
import os, sys
from pathlib import Path
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from src.db import load_env, build_engine
from src.features import build_frame, split_xy
from src.modeling import train_models


## Business KPI Mapping

- Stakeholder owner: Case Management Director
- Decision enabled: prioritize residents for reintegration preparation support
- Primary KPI: successful reintegration progression rate
- Guardrail KPIs: adverse incidents during transition, staff overload index
- Minimum success criteria: +7% readiness progression with no increase in severe incident rate

## Problem Framing
- Predictive: classify residents approaching reintegration readiness.
- Explanatory: identify education, health, and counseling factors associated with readiness.
- This helps prioritize social-worker interventions.

In [ ]:
load_env('.env')
engine = build_engine(os.getenv('DB_PROFILE', 'local'))
df = build_frame(engine)
X, y = split_xy(df)
print(df.shape, 'readiness rate=', round(y.mean(),4))
df.head()

In [ ]:
# Data Understanding Audit: missingness + anomaly checks
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
display(missing_pct.head(10).to_frame('missing_pct'))

audit = {
    'rows': len(df),
    'readiness_positive_rate': float(y.mean()),
    'invalid_attendance_rows': int(((df['attendance_avg'] < 0) | (df['attendance_avg'] > 1)).sum()),
    'invalid_health_rows': int(((df['health_avg'] < 0) | (df['health_avg'] > 5)).sum()),
}
print('Audit summary:', audit)

print('Feature rationale: education, health, and counseling metrics represent reintegration preparation domains.')

In [ ]:
import numpy as np
import subprocess
import sys

try:
    import seaborn as sns
    import matplotlib.pyplot as plt
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'seaborn', 'matplotlib'])
    import seaborn as sns
    import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['edu_progress_avg'], bins=12, kde=True, ax=axes[0])
axes[0].set_title('Education Progress Distribution')

sns.scatterplot(data=df, x='health_avg', y='progress_noted_rate', hue='approaching_readiness', ax=axes[1])
axes[1].set_title('Health vs Counseling Progress Noted')
plt.tight_layout()
plt.show()

In [ ]:
rf, rf_metrics, lg, lg_metrics, coef_df = train_models(X, y)
print('Predictive RF metrics:', {k: round(v,4) for k,v in rf_metrics.items()})
print('Explanatory logistic metrics:', {k: round(v,4) for k,v in lg_metrics.items()})
display(coef_df.head(12))
display(coef_df.tail(12).sort_values('coefficient'))

In [ ]:
# Threshold tuning + FP/FN cost table for readiness classifier
readiness_proba_full = lg.predict_proba(X)[:, 1]
thresholds = np.arange(0.1, 0.95, 0.05)
fp_cost = 1.5
fn_cost = 4.0
rows = []
for t in thresholds:
    pred = (readiness_proba_full >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    total_cost = fp_cost * fp + fn_cost * fn
    rows.append({'threshold': round(float(t), 2), 'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn), 'total_cost': float(total_cost)})

cost_df = pd.DataFrame(rows).sort_values('total_cost').reset_index(drop=True)
display(cost_df.head(10))
best_t = float(cost_df.loc[0, 'threshold'])
print(f'Selected threshold by cost minimization: {best_t:.2f}')

## Evaluation In Business Terms
- False positives: resources allocated to residents not yet ready.
- False negatives: delayed support for residents nearing readiness.
Use outputs to prioritize case conferences and targeted support plans.

## Operationalization Policy + Monitoring

- Threshold policy: use minimum-cost threshold from FP/FN table; default fallback 0.55.
- Action bands: high readiness -> immediate case conference, medium -> targeted support plan, low -> intensified monitoring.
- Retraining cadence: monthly retrain; early retrain if PR-AUC floor breach or data drift alert.
- Monitoring references:
  - `ml-pipelines/integration/pipeline_registry.yaml`
  - `ml-pipelines/integration/monitoring_spec.md`
  - `ml-pipelines/integration/README.md`

## Causal Caveat
Observed associations are not proof of causal readiness drivers.

## Deployment Notes
Surface readiness probability in case-management dashboard with drill-down to top contributing factors.